# TED 02 — Notebook 02: Limpeza e saneamento
Todas as regras estão implementadas em `src/preparacao.py`, para que o mesmo tratamento possa ser reaplicado em novas versões da base (e, no futuro, aos dados da própria Inforcell).

**Decisão estrutural:** o projeto passa a usar **duas bases complementares**:
- **Base A – Reparos** (agregado ORDS v0.3, 4 categorias): contexto do atendimento — aparelho, marca, idade, desfecho e barreira de reparo.
- **Base B – Falhas rotuladas** (quests MobiFix, CompCat e TabiCat): o par **sintoma (texto) → tipo de falha**, que a base agregada não possui e que é o núcleo do assistente de diagnóstico.

In [1]:
import sys
sys.path.insert(0, "../../src")
import pandas as pd, numpy as np
import preparacao as prep
bruto = prep.carregar_bruto()
quests = prep.carregar_quests()

## 1. Base A — Reparos

In [2]:
log_a = []
reparos = prep.limpar_reparos(bruto, log_a)
pd.DataFrame(log_a, columns=["etapa", "registros"])

,etapa,registros
0,Registros brutos (4 categorias),23612
1,"Removidos: itens fora de escopo (GPS, e-reader, estação meteorológica...)",1079
2,Ano de fabricação inválido -> ausente (futuro ou anterior à tecnologia),18
3,Descrições sem conteúdo (placeholder) -> ausente,24
4,Registros idênticos (exceto id) — mantidos e sinalizados,2669
5,Registros na Base A tratada,22533


### 1.1 Justificativas
- **Escopo (remoção):** GPS, e-readers, estações meteorológicas e ciclocomputadores estavam rotulados como *Tablet*. Não são atendidos pela Inforcell e distorceriam as taxas de conserto de tablets → **removidos** (não é possível recategorizá-los dentro das 4 classes).
- **Tipos:** `event_date` → data; `year_of_manufacture`/idade → inteiro anulável (`Int64`), que preserva a ausência sem transformá-la em zero.
- **Ano de fabricação impossível** (posterior ao evento ou anterior à existência da tecnologia): **convertido em ausente**, não removido — o restante do registro (sintoma, desfecho) continua útil.
- **Idade:** `product_age` era redundante com o ano; foi **recalculada** a partir do ano validado para garantir consistência.
- **Idade ausente (~69%): mantida ausente, sem imputação.** Imputar média/mediana criaria uma idade artificial em 2 de cada 3 registros e enfraqueceria justamente a relação idade × desfecho que queremos estudar. As análises de idade usam apenas os registros com valor.
- **Outliers de idade (> 20 anos):** mantidos e sinalizados (`idade_atipica`). São aparelhos vintage reais (ex.: Commodore, Nokia antigos), não erros.
- **Marca:** grafias unificadas (ex.: *appel, iphone, iPad → Apple*; *hp, Hewlett Packard, Compaq → HP*; *Redmi → Xiaomi*). *Unknown* passou a **Desconhecida** (categoria explícita, não imputada).
- **Texto do problema:** normalização de espaços, remoção de URLs e artefatos; placeholders ("?", "unknown", "-") → ausente. Registros sem texto são **mantidos** na Base A (servem para estatísticas de desfecho), mas sinalizados (`tem_descricao`).
- **Idioma:** detectado por palavras funcionais com *fallback* pelo idioma oficial do país (validação na seção 3).
- **Barreira de reparo:** ausência estrutural preenchida com *Não se aplica* (itens que não estão em fim de vida) ou *Não informada* (fim de vida sem barreira registrada).
- **Duplicatas (reparos em lote):** **mantidas** — possuem IDs distintos e representam aparelhos diferentes com o mesmo serviço. São sinalizadas (`registro_em_lote`) para análises de texto.
- **Colunas removidas:** `product_category_id` (1:1 com a categoria), `group_identifier` (mistura códigos e nomes, sem uso analítico), `partner_product_category` (já usada para o filtro de escopo), `product_age` (substituída pela idade recalculada).
- **Status "Unknown":** mantido na base, mas **excluído** do denominador das taxas de conserto (`consertado` = NaN).

In [3]:
reparos.info()

<class 'pandas.DataFrame'>
RangeIndex: 22533 entries, 0 to 22532
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                22533 non-null  str           
 1   fonte             22533 non-null  category      
 2   pais              22533 non-null  category      
 3   categoria         22533 non-null  category      
 4   marca             22533 non-null  category      
 5   ano_fabricacao    6867 non-null   Int64         
 6   idade_anos        6867 non-null   Int64         
 7   faixa_idade       6867 non-null   category      
 8   idade_atipica     22533 non-null  bool          
 9   status_reparo     22533 non-null  category      
 10  consertado        21694 non-null  float64       
 11  barreira_reparo   22533 non-null  category      
 12  data_evento       22533 non-null  datetime64[us]
 13  ano_evento        22533 non-null  Int64         
 14  problema          16982 non-null 

In [4]:
prep.perfil_qualidade(reparos)

,tipo,nao_nulos,ausentes,%_ausentes,valores_distintos
id,str,22533,0,0.0,22533
fonte,category,22533,0,0.0,8
pais,category,22533,0,0.0,28
categoria,category,22533,0,0.0,4
marca,category,22533,0,0.0,685
ano_fabricacao,Int64,6867,15666,69.5,42
idade_anos,Int64,6867,15666,69.5,37
faixa_idade,category,6867,15666,69.5,5
idade_atipica,bool,22533,0,0.0,2
status_reparo,category,22533,0,0.0,4


In [5]:
reparos.sample(6, random_state=11)[["categoria", "marca", "idade_anos", "status_reparo", "idioma", "problema"]]

,categoria,marca,idade_anos,status_reparo,idioma,problema
9021,Notebook,Sony,<NA>,Consertado,en,Slow
8102,Notebook,Apple,<NA>,Consertado,en,"battery problems. turns out it's 5 years old, still OK for the age, taught how to reduce battery usage and check performance"
7598,Notebook,Lenovo,4,Consertado,en,no issues. W253
7050,Desktop,Desconhecida,<NA>,Reparável (não concluído),sem texto,NaN
16841,Celular,Apple,10,Consertado,en,Broken screen
12191,Notebook,Acer,8,Fim de vida,en,Tried to fix but screen would not turn on. wouldn't boot to hub/live USB. Graphics card had power. Took out HDD/DVD to re-use


## 2. Base B — Falhas rotuladas (sintoma → tipo de falha)

In [6]:
for nome, q in quests.items():
    col = "fault_type"
    print(f"{nome:17s} {q.shape}  tipos de falha distintos: {q[col].nunique()}")

mobifix           (1916, 11)  tipos de falha distintos: 25
compcat_laptops   (3511, 11)  tipos de falha distintos: 20
compcat_desktops  (583, 11)  tipos de falha distintos: 19
tabicat           (647, 12)  tipos de falha distintos: 22


In [7]:
log_b = []
falhas = prep.montar_falhas(quests, log_b)
pd.DataFrame(log_b, columns=["etapa", "registros"])

,etapa,registros
0,Registros rotulados brutos (MobiFix + CompCat + TabiCat),6657
1,Ano de fabricação inválido -> ausente,1
2,Duplicatas de texto+rótulo (sinalizadas),1016
3,"Registros aptos para modelagem (texto + rótulo informativo, sem duplicata)",4019


### 2.1 Harmonização da taxonomia de falhas
As três quests usam listas diferentes (MobiFix/TabiCat: 26 tipos; CompCat: 20 tipos). Os 41 rótulos originais distintos foram mapeados em **14 categorias** em português, próximas da rotina de bancada da Inforcell e das etapas do diagnóstico guiado. *Unknown* e *Poor data* viraram **Indeterminado** (rótulo não informativo).

In [8]:
mapa = (falhas.groupby(["categoria_falha", "tipo_falha_original"], observed=True).size()
        .rename("registros").reset_index().sort_values(["categoria_falha", "registros"], ascending=[True, False]))
mapa

,categoria_falha,tipo_falha_original,registros
0,Armazenamento,Internal storage,210
1,Armazenamento,Storage problem,47
2,Botões / teclado / touchpad,Integrated keyboard,112
4,Botões / teclado / touchpad,On/Off button,44
5,Botões / teclado / touchpad,Other buttons,41
3,Botões / teclado / touchpad,Integrated pointing device,25
6,Botões / teclado / touchpad,Volume buttons,8
7,Carcaça / dano externo,Case/chassis,80
8,Carcaça / dano externo,External damage,6
13,Conectores e portas,USB/charging port,142


### 2.2 Justificativas
- **`????` no ano (TabiCat):** marcador textual de ausência → convertido em nulo.
- **Idade × ano (MobiFix):** a MobiFix traz idade em vez de ano; convertemos para ano de fabricação para padronizar com as demais.
- **Rótulo Indeterminado:** mantido na base descritiva (mostra a dificuldade real de diagnóstico), mas **fora do conjunto de modelagem**.
- **Registros sem texto:** não servem para aprender sintoma → causa; ficam **fora do conjunto de modelagem**.
- **Duplicatas de texto + rótulo:** mantidas na base descritiva, **fora do conjunto de modelagem** — evita que o futuro modelo decore frases repetidas e superestime sua acurácia.
- A flag `apto_modelagem` consolida essas regras para a próxima TED.

In [9]:
falhas["apto_modelagem"].value_counts()

apto_modelagem
True     4019
False    2638

## 3. Validação do detector de idioma
A TabiCat possui o idioma rotulado manualmente — usamos esses rótulos como gabarito.

In [10]:
tab = falhas[falhas["quest"] == "TabiCat"].copy()
tab["idioma_real"] = quests["tabicat"]["language"].values
det = tab[~tab["idioma"].isin(["indefinido", "sem texto"])]
print(f"Cobertura: {len(det)/len(tab):.1%} | Acurácia nos detectados: {(det['idioma'] == det['idioma_real']).mean():.1%}")
pd.crosstab(tab["idioma_real"], tab["idioma"])

Cobertura: 96.0% | Acurácia nos detectados: 96.5%


idioma,de,en,fr,indefinido,it,nl,no,sem texto
idioma_real,,,,,,,,
de,10,0,0,0,0,1,0,0
en,1,535,0,23,7,4,7,0
fr,0,1,3,0,0,0,0,0
it,0,0,0,0,4,0,0,0
nl,0,1,0,1,0,47,0,2


## 4. Base após o tratamento e exportação

In [11]:
resumo = pd.DataFrame({
    "Base": ["A – Reparos", "B – Falhas rotuladas", "B – apta para modelagem"],
    "Registros": [len(reparos), len(falhas), int(falhas["apto_modelagem"].sum())],
    "Colunas": [reparos.shape[1], falhas.shape[1], falhas.shape[1]],
    "Com descrição (%)": [round(reparos["tem_descricao"].mean()*100, 1), round(falhas["tem_descricao"].mean()*100, 1), 100.0],
})
resumo

,Base,Registros,Colunas,Com descrição (%)
0,A – Reparos,22533,19,75.4
1,B – Falhas rotuladas,6657,20,82.6
2,B – apta para modelagem,4019,20,100.0


In [12]:
prep.PROCESSED.mkdir(parents=True, exist_ok=True)
reparos.to_csv(prep.PROCESSED / "reparos_limpo.csv", index=False)
falhas.to_csv(prep.PROCESSED / "falhas_rotuladas_limpo.csv", index=False)
falhas[falhas["apto_modelagem"]][["id_origem", "categoria", "marca", "idioma", "problema", "categoria_falha", "status_reparo"]] \
    .to_csv(prep.PROCESSED / "falhas_modelagem.csv", index=False)
mapa.to_csv(prep.PROCESSED / "taxonomia_falhas.csv", index=False)
for f in sorted(prep.PROCESSED.glob("*.csv")):
    print(f"{f.name:32s} {f.stat().st_size/1024:8.0f} KB")

falhas_modelagem.csv                  486 KB
falhas_rotuladas_limpo.csv           1251 KB
reparos_limpo.csv                    4082 KB
taxonomia_falhas.csv                    2 KB
